# Chapter 15 — Is Similarity One-Dimensional?

**Book alignment:** Embeddings From First Principles, Chapter 15

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Two pairs, identical cosine 0.78, different situations
(one a confident isolated match, one a coin toss in a hub). How much of its own uncertainty
can the *geometry alone* diagnose before a second model is called? On RELATE hard negatives,
six geometric signals lift balanced accuracy from 0.76 to 0.90; an NLI cross-encoder on top
adds **nothing** (the committed Wave 1 ablation).

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Same scalar, different situation — the signals the cosine threw away

In [ ]:
D = 48
def unit(v): return v / np.linalg.norm(v)
q = unit(rng.standard_normal(D))

def at_cos(target):                       # a unit vector at a prescribed cosine to q
    r = rng.standard_normal(D)
    perp = r - (r @ q) * q
    perp = perp / np.linalg.norm(perp)
    return target * q + np.sqrt(1 - target ** 2) * perp

cand1 = [at_cos(c) for c in (0.78, 0.44, 0.42, 0.40)]   # isolated match
cand2 = [at_cos(c) for c in (0.78, 0.77, 0.76, 0.76)]   # a hub: near-ties behind the leader

def margin(cs):
    s = sorted((float(q @ c) for c in cs), reverse=True)
    return s[0], s[0] - s[1]

m1, m2 = margin(cand1), margin(cand2)
print(f"pair 1: top={m1[0]:.2f}  margin={m1[1]:+.3f}  -> confident, isolated")
print(f"pair 2: top={m2[0]:.2f}  margin={m2[1]:+.3f}  -> coin toss dressed as a match")
assert abs(m1[0] - m2[0]) < 0.01 and m1[1] > m2[1] + 0.2
print("the scalar is identical; the margin is not")

## 2. Geometry diagnoses its own hard cases — measured on RELATE (Wave 1)

In [ ]:
sa = art("wave1", "signal-ablation.json")
acc = sa["accuracy"]
print("balanced accuracy separating grade-3 positive from hard negatives:")
print(f"  score only                       {acc['score_only']:.3f}")
print(f"  + margin,density,topk_std,in_deg,rank  {acc['geometric']:.3f}   (+{sa['geometry_lift_over_score']:.3f})")
print(f"  + NLI cross-encoder              {acc['geometric_plus_nli']:.3f}   ({sa['nli_increment']:+.4f})")

assert acc["geometric"] - acc["score_only"] > 0.12       # geometry recovers +0.14 on the hard cases
assert abs(sa["nli_increment"]) < 0.01                   # a generic second model adds nothing
print(f"\nfeatures: {sa['geometric_features']}")
print("the phenomenon 'is this the right match?' is better described by 5-6 numbers than by 1")
print("the external increment depends on the verifier being matched to the missing distinction")

## What we earned

A similarity scalar discards geometric signal — margin, local density, hubness, rank
stability, top-k spread — that is decision-relevant on hard cases. On RELATE those six
geometric signals lifted correct-match balanced accuracy from 0.76 to 0.90 with **no second
model called**; a generic NLI cross-encoder on top added −0.003. The diagnostic vector pays
off when errors are expensive and the easy/hard gap is wide.

**Notebook 16 / Chapter 16** introduces a second model — and finds the same text, embedded
by a different encoder, enters a different coordinate system.